In [1]:
!pip install pandas requests


Active code page: 1252
Defaulting to user installation because normal site-packages is not writeable


In [2]:
import os
import pandas as pd
import requests


In [3]:
# Absolute path (safe option for you)
RAW_DATA_PATH = r"C:\Users\satya\Downloads\JoshTalks_ASR_Assignment\task\data\raw_inputs\FT Data - data.csv"

df = pd.read_csv(RAW_DATA_PATH)
df.columns = df.columns.str.strip()

print("Columns Found:")
print(df.columns.tolist())

df.head(3)


Columns Found:
['user_id', 'recording_id', 'language', 'duration', 'rec_url_gcp', 'transcription_url_gcp', 'metadata_url_gcp']


,user_id,recording_id,language,duration,rec_url_gcp,transcription_url_gcp,metadata_url_gcp
0,245746,825780,hi,443,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...
1,291038,825727,hi,443,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...
2,246004,988596,hi,475,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...


In [4]:
def transform_url(wrong_url):
    """
    Convert:
    https://storage.googleapis.com/joshtalks-data-collection/hq_data/hi/967179/825780_audio.wav
    
    To:
    https://storage.googleapis.com/upload_goai/967179/825780_audio.wav
    """
    parts = wrong_url.split("/")
    
    folder_id = parts[-2]
    filename = parts[-1]
    
    correct_url = f"https://storage.googleapis.com/upload_goai/{folder_id}/{filename}"
    
    return correct_url


In [5]:
sample_audio = df.loc[0, "rec_url_gcp"]
sample_trans = df.loc[0, "transcription_url_gcp"]

fixed_audio = transform_url(sample_audio)
fixed_trans = transform_url(sample_trans)

print("Original Audio:\n", sample_audio)
print("\nFixed Audio:\n", fixed_audio)

print("\nOriginal Transcript:\n", sample_trans)
print("\nFixed Transcript:\n", fixed_trans)


Original Audio:
 https://storage.googleapis.com/joshtalks-data-collection/hq_data/hi/967179/825780_audio.wav

Fixed Audio:
 https://storage.googleapis.com/upload_goai/967179/825780_audio.wav

Original Transcript:
 https://storage.googleapis.com/joshtalks-data-collection/hq_data/hi/967179/825780_transcription.json

Fixed Transcript:
 https://storage.googleapis.com/upload_goai/967179/825780_transcription.json


In [6]:
response = requests.get(fixed_audio)

print("Status Code:", response.status_code)
print("File Size (bytes):", len(response.content))


Status Code: 200
File Size (bytes): 34307062


In [7]:
# Apply transformation to full dataset
df["fixed_audio_url"] = df["rec_url_gcp"].apply(transform_url)
df["fixed_trans_url"] = df["transcription_url_gcp"].apply(transform_url)

df.head()


,user_id,recording_id,language,duration,rec_url_gcp,transcription_url_gcp,metadata_url_gcp,fixed_audio_url,fixed_trans_url
0,245746,825780,hi,443,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/upload_goai/967...,https://storage.googleapis.com/upload_goai/967...
1,291038,825727,hi,443,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/upload_goai/967...,https://storage.googleapis.com/upload_goai/967...
2,246004,988596,hi,475,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/upload_goai/114...,https://storage.googleapis.com/upload_goai/114...
3,93626,990175,hi,475,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/upload_goai/114...,https://storage.googleapis.com/upload_goai/114...
4,286851,526266,hi,522,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/upload_goai/639...,https://storage.googleapis.com/upload_goai/639...


In [8]:
UPDATED_CSV_PATH = r"C:\Users\satya\Downloads\JoshTalks_ASR_Assignment\task\data\raw_inputs\FT_Data_with_fixed_urls.csv"

df.to_csv(UPDATED_CSV_PATH, index=False)

print("Updated CSV saved at:", UPDATED_CSV_PATH)


Updated CSV saved at: C:\Users\satya\Downloads\JoshTalks_ASR_Assignment\task\data\raw_inputs\FT_Data_with_fixed_urls.csv
